In [ ]:
from bauer.utils.data import load_garcia2022
from bauer.models import MagnitudeComparisonModel
import numpy as np
import pandas as pd
import seaborn as sns
from bauer.utils.plotting import plot_ppc
import os.path as op

df = load_garcia2022()
target_folder ='/Users/mrenke/data/ds-miguel/derivatives/phenotype'



WARNING (pytensor.tensor.blas): Using NumPy C-API based implementation for BLAS functions.


### hierachical 


In [ ]:
model_label = 3 
model = MagnitudeComparisonModel(df, 
                            fit_prior=False, # model_label = 3, True --> model_label = 5
                            fit_seperate_evidence_sd = True, 
                            memory_model='shared_perceptual_noise',
                            )

model.build_estimation_model()
trace = model.sample(3000, 2000, target_accept=0.95)

In [ ]:
from bauer.utils.bayes import softplus

def get_subwise_params(idata,param_name):
    df_param= idata.posterior[param_name].to_dataframe()    
    df_param.columns.name = 'parameter'
    df_param.index = df_param.index.set_names(['chain','draw','subject'])
    df_param = df_param.stack().to_frame('value')
    df_param = df_param.groupby(['subject'])[['value']].mean()
    df_param = df_param.rename(mapper={'value':param_name},axis=1)
    if 'sd' in param_name:
        df_param[param_name]= softplus(df_param[param_name]) 
    return df_param


In [ ]:
sd_mem = get_subwise_params(trace, 'memory_noise_sd')
sd_per = get_subwise_params(trace, 'perceptual_noise_sd')
sds = pd.concat([sd_mem, sd_per], axis=1)

sds.to_csv(op.join(target_folder, f'magjudge_bauer-{model_label}_sds.csv')) #_untransformed

### unbiased

In [ ]:
model_label = 3 
model = MagnitudeComparisonModel(df, 
                            fit_prior=False, # model_label = 3, True --> model_label = 5
                            fit_seperate_evidence_sd = True, 
                            memory_model='shared_perceptual_noise',
                            )

def fit_model_individual(d, model=model):
    model.build_estimation_model(d, hierarchical=False)
    return model.fit_map()

maps = df.groupby(['subject']).apply(fit_model_individual)


In [ ]:
from bauer.utils.bayes import softplus

#maps = softplus(maps) # think I do not need this, looks like theya re already transformed, maybe because no group level?
df_maps = pd.DataFrame(maps.tolist(), index=maps.index)
df_maps.to_csv(op.join(target_folder, f'bauer-{model_label}_sds-maps_unbiased.csv'))